# Analisi dei risultati — edizione grafica per la tesi

Versione consolidata e ristilizzata di `04_colab_results_analysis.ipynb`.
Legge **soltanto** i CSV già prodotti in `final_analysis/tables`: non apre gli archivi delle run
e non importa librerie di training o simulazione quantistica.

Output: cinque figure coordinate in PNG/PDF, didascalie e manifest di riproducibilità.


In [ ]:
# Percorsi: normalmente non serve modificare altro.
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

DEFAULT_ROOT = (
    Path("/content/drive/MyDrive/tesi_qml_colab") if IN_COLAB else Path.cwd()
)
INPUT_DIR = Path(os.environ.get(
    "TESI_ANALYSIS_INPUT", DEFAULT_ROOT / "final_analysis"
))
TABLE_DIR = INPUT_DIR / "tables"
OUTPUT_DIR = Path(os.environ.get(
    "TESI_RESTYLED_OUTPUT", DEFAULT_ROOT / "final_analysis_restyled"
))
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FAST_MODE = os.environ.get("TESI_ANALYSIS_FAST", "0") == "1"
EXPORT_DPI = 100 if FAST_MODE else 300
REQUIRED_TABLES = (
    "q_minus_b2_paired.csv", "q_minus_b2_holm_tests.csv", "q_minus_qf.csv",
    "binary_to_three_class.csv", "merger_detailed.csv", "qubit_8_minus_4.csv",
    "cost_ratios.csv",
)
missing = [name for name in REQUIRED_TABLES if not (TABLE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"CSV mancanti in {TABLE_DIR}: {missing}. "
        "Esegui prima 04_colab_results_analysis.ipynb."
    )

print({"input_tables": str(TABLE_DIR), "output": str(OUTPUT_DIR), "dpi": EXPORT_DPI})


In [ ]:
# Solo stack di analisi dati e grafica.
import json
import math
import sys
import zlib
from datetime import datetime, timezone

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter, FuncFormatter
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "axes.labelsize": 10.5,
    "axes.labelweight": "medium",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#374151",
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": EXPORT_DPI,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "legend.frameon": False,
    "axes.axisbelow": True,
})

PALETTE = {
    "B0": "#9CA3AF", "B1": "#4B5563", "B2": "#1E3A8A",
    "Q": "#7C3AED", "QF": "#D97706", "significant": "#DC2626",
    "positive": "#1E3A8A", "neutral": "#6B7280", "grid": "#E5E7EB",
}
MODEL_STYLES = {
    "B2": {"marker": "o", "linestyle": "-", "label": "B2 · controllo classico"},
    "Q": {"marker": "D", "linestyle": "-", "label": "Q · quantistico addestrabile"},
    "QF": {"marker": "^", "linestyle": "--", "label": "QF · circuito congelato"},
}
TASK_LABELS = {"binary": "Task binario", "three_class": "Task a tre classi (+ merger)"}
REGIMES = ("100", "300", "900", "full")
REGIME_LABELS = {"100": "100", "300": "300", "900": "900", "full": "Full"}
FULL_SAMPLES = {"binary": 1876, "three_class": 2814}
EXPORTED_FIGURES = []


## Preparazione dei dati dai CSV


In [ ]:
def normalize_regime(value):
    text = str(value).strip().lower()
    return text[:-2] if text.endswith(".0") else text


def load_table(name):
    frame = pd.read_csv(TABLE_DIR / f"{name}.csv")
    if "subset_size" in frame:
        frame["subset_size"] = frame["subset_size"].map(normalize_regime)
    return frame


q_b2 = load_table("q_minus_b2_paired")
q_qf = load_table("q_minus_qf")
holm = load_table("q_minus_b2_holm_tests")
task_transition = load_table("binary_to_three_class")
merger = load_table("merger_detailed")
qubit_delta = load_table("qubit_8_minus_4")
costs = load_table("cost_ratios")

key = ["task", "subset_size", "n_qubits", "repeat_seed", "fold"]
performance_parts = [
    q_b2[key + ["B2"]].rename(columns={"B2": "f1_macro"}).assign(model="B2"),
    q_b2[key + ["Q"]].rename(columns={"Q": "f1_macro"}).assign(model="Q"),
    q_qf[key + ["QF"]].rename(columns={"QF": "f1_macro"}).assign(model="QF"),
]
performance = pd.concat(performance_parts, ignore_index=True)
performance["training_samples"] = [
    FULL_SAMPLES[row.task] if row.subset_size == "full" else int(row.subset_size)
    for row in performance.itertuples()
]
performance = performance.drop_duplicates(key + ["model"])

expected_cells = 2 * 4 * 2 * 3
cell_counts = performance.groupby(
    ["task", "subset_size", "n_qubits", "model"], dropna=False
)["repeat_seed"].nunique()
if len(cell_counts) != expected_cells or not cell_counts.eq(10).all():
    raise ValueError(
        "Protocollo CSV inatteso: servono 48 celle B2/Q/QF con 10 seed ciascuna; "
        f"osservate {len(cell_counts)} celle e seed {cell_counts.min()}–{cell_counts.max()}."
    )

for frame in (q_b2, q_qf, task_transition, merger, qubit_delta, costs, holm):
    if "n_qubits" in frame:
        frame["n_qubits"] = pd.to_numeric(frame["n_qubits"], errors="coerce")

display(pd.DataFrame({
    "Tabella": ["Prestazioni ricostruite", "Q−B2 appaiato", "Costi appaiati"],
    "Righe": [len(performance), len(q_b2), len(costs)],
}))


In [ ]:
def stable_seed(*parts):
    return (20260905 + zlib.crc32("|".join(map(str, parts)).encode())) % (2**32)


def interval(values, label=""):
    clean = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(float)
    estimate = float(np.mean(clean))
    if len(clean) < 2:
        return estimate, estimate, estimate
    repetitions = 300 if FAST_MODE else 10_000
    rng = np.random.default_rng(stable_seed(label))
    means = rng.choice(clean, size=(repetitions, len(clean)), replace=True).mean(axis=1)
    low, high = np.quantile(means, [0.025, 0.975])
    return estimate, float(low), float(high)


def summary(frame, groups, value):
    rows = []
    for group_key, group in frame.groupby(groups, observed=True, sort=False, dropna=False):
        keys = group_key if isinstance(group_key, tuple) else (group_key,)
        estimate, low, high = interval(group[value], (*keys, value))
        rows.append({**dict(zip(groups, keys)), "estimate": estimate,
                     "ci95_low": low, "ci95_high": high, "n": len(group)})
    return pd.DataFrame(rows)


def clean_axis(ax, grid=True):
    if grid:
        ax.grid(True, axis="y", color=PALETTE["grid"], linewidth=0.55)
    else:
        ax.grid(False, axis="y")
    ax.grid(False, axis="x")
    ax.tick_params(length=3, color="#6B7280")


def panel_label(ax, text):
    ax.text(-0.08, 1.05, text, transform=ax.transAxes, fontsize=12,
            fontweight="bold", va="bottom", color="#111827")


def save_figure(fig, name):
    paths = {}
    for suffix in ("png", "pdf"):
        path = FIGURE_DIR / f"{name}.{suffix}"
        kwargs = {"dpi": EXPORT_DPI} if suffix == "png" else {}
        fig.savefig(path, bbox_inches="tight", facecolor="white", **kwargs)
        paths[suffix] = str(path)
    display(fig)
    plt.close(fig)
    EXPORTED_FIGURES.append({"name": name, **paths})
    print(f"Salvati: {name}.png, {name}.pdf")


def model_legend():
    return [Line2D([0], [0], color=PALETTE[m], marker=MODEL_STYLES[m]["marker"],
                   linestyle=MODEL_STYLES[m]["linestyle"], linewidth=2,
                   markeredgecolor="white", label=MODEL_STYLES[m]["label"])
            for m in ("B2", "Q", "QF")]


# Figura 1 — Scaling delle prestazioni

Asse categoriale esplicito, una sola traiettoria media per modello e IC95% bootstrap.
I seed restano l'unità statistica, ma non vengono collegati individualmente.


In [ ]:
hero = performance.loc[performance["n_qubits"].eq(8)].copy()
hero_summary = summary(
    hero, ["task", "model", "subset_size", "training_samples"], "f1_macro"
)
x_positions = np.arange(len(REGIMES))
fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.0))

for panel_index, task in enumerate(("binary", "three_class")):
    ax = axes[panel_index]
    task_summary = hero_summary.loc[hero_summary["task"].eq(task)].copy()
    for model in ("B2", "Q", "QF"):
        rows = task_summary.loc[task_summary["model"].eq(model)].copy()
        rows["position"] = rows["subset_size"].map({r: i for i, r in enumerate(REGIMES)})
        rows = rows.sort_values("position")
        mean = rows["estimate"].to_numpy(float)
        low = rows["ci95_low"].to_numpy(float)
        high = rows["ci95_high"].to_numpy(float)
        ax.fill_between(x_positions, low, high, color=PALETTE[model], alpha=0.12,
                        linewidth=0, zorder=1)
        ax.plot(x_positions, mean, color=PALETTE[model], linewidth=2.4,
                linestyle=MODEL_STYLES[model]["linestyle"],
                marker=MODEL_STYLES[model]["marker"], markersize=7,
                markeredgecolor="white", markeredgewidth=1.0, zorder=2)

    full = task_summary.loc[task_summary["subset_size"].eq("full")]
    b2_mean = float(full.loc[full["model"].eq("B2"), "estimate"].iloc[0])
    q_mean = float(full.loc[full["model"].eq("Q"), "estimate"].iloc[0])
    test = holm.loc[
        holm["task"].eq(task) & holm["subset_size"].eq("full") & holm["n_qubits"].eq(8)
    ]
    p_holm = float(test["holm_p_value"].iloc[0]) if len(test) else np.nan
    annotation_color = PALETTE["significant"] if p_holm < 0.05 else PALETTE["neutral"]
    ax.text(0.98, 0.04,
            f"Full: Q − B2 = {100 * (q_mean - b2_mean):+.2f} pp  ·  p Holm = {p_holm:.3g}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=8.5,
            color=annotation_color, fontweight="semibold")

    ax.set_xticks(range(4), [REGIME_LABELS[r] for r in REGIMES])
    ax.set_xlim(-0.18, 3.18)
    ax.set_xlabel("Immagini di training")
    ax.set_ylabel("Macro-F1")
    ax.set_title(TASK_LABELS[task], loc="left", pad=10)
    values = hero.loc[hero["task"].eq(task), "f1_macro"]
    padding = max(0.012, 0.13 * (values.max() - values.min()))
    ax.set_ylim(max(0, values.min() - padding), min(1, values.max() + padding))
    clean_axis(ax)
    panel_label(ax, chr(ord("a") + panel_index))

fig.legend(handles=model_legend(), loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=3)
fig.suptitle("L'aumento dei dati separa il controllo classico dal modello quantistico",
             fontsize=14, fontweight="semibold", y=1.10)
fig.text(0.5, -0.015,
         "Punti: media sui 10 seed · bande: IC95% bootstrap · configurazioni a 8 qubit",
         ha="center", fontsize=8.8, color="#4B5563")
fig.tight_layout()
save_figure(fig, "01_hero_learning_curves")


# Figura 2 — Differenziale appaiato Q−B2

Diverging bar plot compatto. Ogni barra riassume i dieci seed appaiati; lo zero indica
parità tra il controllo classico e il modello quantistico.


In [ ]:
delta = q_b2.copy()
if "difference_pp" not in delta:
    delta["difference_pp"] = 100 * delta["difference"]
delta_summary = summary(delta, ["task", "subset_size", "n_qubits"], "difference_pp")
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.8))

for panel_index, task in enumerate(("binary", "three_class")):
    ax = axes[panel_index]
    plot_rows = []
    for regime in REGIMES:
        for qubits in (4, 8):
            row = delta_summary.loc[
                delta_summary["task"].eq(task)
                & delta_summary["subset_size"].eq(regime)
                & delta_summary["n_qubits"].eq(qubits)
            ].iloc[0].to_dict()
            test = holm.loc[
                holm["task"].eq(task) & holm["subset_size"].eq(regime)
                & holm["n_qubits"].eq(qubits)
            ]
            row["p_holm"] = float(test["holm_p_value"].iloc[0]) if len(test) else np.nan
            row["label"] = f"{REGIME_LABELS[regime]} · {qubits}q"
            plot_rows.append(row)
    plot_frame = pd.DataFrame(plot_rows)
    y = np.arange(len(plot_frame))
    significant = plot_frame["p_holm"].lt(0.05)
    colors = np.where(
        significant & plot_frame["estimate"].lt(0), PALETTE["significant"],
        np.where(significant, PALETTE["positive"], PALETTE["neutral"]),
    )
    ax.barh(y, plot_frame["estimate"], color=colors, height=0.58, alpha=0.88, zorder=2)
    left_error = plot_frame["estimate"] - plot_frame["ci95_low"]
    right_error = plot_frame["ci95_high"] - plot_frame["estimate"]
    ax.errorbar(plot_frame["estimate"], y, xerr=np.vstack([left_error, right_error]),
                fmt="none", ecolor="#111827", elinewidth=0.9, capsize=2.5, zorder=3)
    span = max(abs(plot_frame["ci95_low"].min()), abs(plot_frame["ci95_high"].max()), 0.5)
    ax.set_xlim(-1.25 * span, 1.25 * span)
    label_offset = 0.035 * (2.5 * span)
    for row_index, row in plot_frame.iterrows():
        star = "  *" if row["p_holm"] < 0.05 else ""
        ha = "left" if row["estimate"] >= 0 else "right"
        offset = label_offset if row["estimate"] >= 0 else -label_offset
        ax.text(row["estimate"] + offset, row_index, f"{row['estimate']:+.2f}{star}",
                va="center", ha=ha, fontsize=8,
                color=PALETTE["significant"] if star else "#374151",
                fontweight="semibold" if star else "normal")
    ax.axvline(0, color="#374151", linestyle=(0, (3, 3)), linewidth=1.0, zorder=1)
    ax.text(0.98, 0.98, "Parità classico–quantistico", transform=ax.transAxes,
            ha="right", va="top", fontsize=8.2, color="#4B5563")
    ax.set_yticks(y, plot_frame["label"])
    ax.invert_yaxis()
    ax.set_xlabel("Δ Macro-F1: Q − B2 (punti percentuali)")
    ax.set_title(TASK_LABELS[task], loc="left")
    clean_axis(ax, grid=False)
    ax.grid(True, axis="x", color=PALETTE["grid"], linewidth=0.55)
    panel_label(ax, chr(ord("a") + panel_index))

fig.suptitle("Il differenziale appaiato identifica i regimi in cui Q perde terreno",
             fontsize=14, fontweight="semibold", y=1.02)
fig.text(0.5, -0.015, "Barra: media · baffi: IC95% bootstrap · * p Holm < 0,05",
         ha="center", fontsize=8.8, color="#4B5563")
fig.tight_layout()
save_figure(fig, "02_delta_q_minus_b2")


# Figura 3 — Differenza-in-differenze tra i task

La quantità visualizzata è
$[(Q-B2)_{\mathrm{tre\ classi}}-(Q-B2)_{\mathrm{binario}}]$.
In questo modo la caduta comune a tutti i modelli non nasconde la penalità relativa di Q.


In [ ]:
task_delta = delta.pivot(
    index=["subset_size", "n_qubits", "repeat_seed", "fold"],
    columns="task", values="difference_pp",
).dropna(subset=["binary", "three_class"]).reset_index()
task_delta["difference_in_difference_pp"] = task_delta["three_class"] - task_delta["binary"]
did_summary = summary(
    task_delta, ["subset_size", "n_qubits"], "difference_in_difference_pp"
)

fig, ax = plt.subplots(figsize=(8.8, 5.0))
for qubits, marker, color in ((4, "o", "#4B5563"), (8, "D", PALETTE["Q"])):
    rows = did_summary.loc[did_summary["n_qubits"].eq(qubits)].copy()
    rows["position"] = rows["subset_size"].map({r: i for i, r in enumerate(REGIMES)})
    rows = rows.sort_values("position")
    x = rows["position"].to_numpy(float)
    mean = rows["estimate"].to_numpy(float)
    low = rows["ci95_low"].to_numpy(float)
    high = rows["ci95_high"].to_numpy(float)
    ax.fill_between(x, low, high, color=color, alpha=0.10, linewidth=0)
    ax.plot(x, mean, color=color, marker=marker, linewidth=2.0, markersize=7,
            markeredgecolor="white", label=f"{qubits} qubit")
ax.axhline(0, color="#374151", linestyle=(0, (3, 3)), linewidth=1.0)
ax.set_xticks(range(4), [REGIME_LABELS[r] for r in REGIMES])
ax.set_xlim(-0.18, 3.18)
ax.set_ylim(-5, 1)
ax.set_xlabel("Immagini di training")
ax.set_ylabel("Differenza-in-differenze (punti percentuali)")
ax.set_title("Q perde terreno quando il task passa da binario a tre classi", loc="left", pad=10)
ax.legend(loc="lower left")
clean_axis(ax)
fig.text(0.5, -0.02,
         "Valori negativi: la penalità Q−B2 è maggiore nel task a tre classi · media e IC95% bootstrap",
         ha="center", fontsize=8.8, color="#4B5563")
fig.tight_layout()
save_figure(fig, "03_binary_to_three_class")


# Figura 4 — Classe merger

La difficoltà della terza classe viene isolata in una figura autonoma, con un pannello per
numero di qubit e senza sovrapporre analisi di ablation.


In [ ]:
merger_f1 = merger.loc[merger["value_column"].eq("merger_f1")].copy()
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), sharey=True)
for panel_index, qubits in enumerate((4, 8)):
    ax = axes[panel_index]
    panel = merger_f1.loc[merger_f1["n_qubits"].eq(qubits)].copy()
    for model in ("B2", "Q", "QF"):
        rows = panel.loc[panel["model"].eq(model)].copy()
        rows["position"] = rows["subset_size"].map({r: i for i, r in enumerate(REGIMES)})
        rows = rows.sort_values("position")
        x = rows["position"].to_numpy(float)
        low = rows["ci95_low"].to_numpy(float)
        high = rows["ci95_high"].to_numpy(float)
        ax.fill_between(x, low, high, color=PALETTE[model], alpha=0.11, linewidth=0)
        ax.plot(x, rows["estimate"], color=PALETTE[model], linewidth=2.1,
                marker=MODEL_STYLES[model]["marker"],
                linestyle=MODEL_STYLES[model]["linestyle"], markersize=6.5,
                markeredgecolor="white")
    ax.set_xticks(range(4), [REGIME_LABELS[r] for r in REGIMES])
    ax.set_xlim(-0.18, 3.18)
    ax.set_xlabel("Immagini di training")
    ax.set_title(f"{qubits} qubit", loc="left")
    clean_axis(ax)
    panel_label(ax, chr(ord("a") + panel_index))
axes[0].set_ylabel("F1 della classe merger")
fig.legend(handles=model_legend(), loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=3)
fig.suptitle("La classe merger concentra la separazione tra controllo classico e quantistico",
             fontsize=14, fontweight="semibold", y=1.10)
fig.text(0.5, -0.02, "Punti: media sui 10 seed · bande: IC95% bootstrap",
         ha="center", fontsize=8.8, color="#4B5563")
fig.tight_layout()
save_figure(fig, "04_merger_detailed")


# Figura 5 — Ablation Q−QF

Il contributo dell'addestramento dei parametri quantistici è mostrato come effetto divergente,
separato sia dalla classe merger sia dal confronto 8−4 qubit.


In [ ]:
q_qf_delta = q_qf.copy()
if "difference_pp" not in q_qf_delta:
    q_qf_delta["difference_pp"] = 100 * q_qf_delta["difference"]
q_qf_summary = summary(
    q_qf_delta, ["task", "subset_size", "n_qubits"], "difference_pp"
)
fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.5), sharey=True)
for panel_index, task in enumerate(("binary", "three_class")):
    ax = axes[panel_index]
    rows = []
    for regime in REGIMES:
        for qubits in (4, 8):
            row = q_qf_summary.loc[
                q_qf_summary["task"].eq(task) & q_qf_summary["subset_size"].eq(regime)
                & q_qf_summary["n_qubits"].eq(qubits)
            ].iloc[0].to_dict()
            row["label"] = f"{REGIME_LABELS[regime]} · {qubits}q"
            rows.append(row)
    frame = pd.DataFrame(rows)
    y = np.arange(len(frame))
    colors = np.where(frame["estimate"].ge(0), PALETTE["Q"], PALETTE["QF"])
    ax.barh(y, frame["estimate"], color=colors, height=0.58, alpha=0.88)
    ax.errorbar(frame["estimate"], y,
                xerr=np.vstack([frame["estimate"] - frame["ci95_low"],
                                frame["ci95_high"] - frame["estimate"]]),
                fmt="none", ecolor="#111827", elinewidth=0.9, capsize=2.5)
    limit = 1.25 * max(abs(frame["ci95_low"].min()), abs(frame["ci95_high"].max()), 0.25)
    ax.set_xlim(-limit, limit)
    ax.axvline(0, color="#374151", linestyle=(0, (3, 3)), linewidth=1.0)
    ax.set_yticks(y, frame["label"])
    ax.invert_yaxis()
    ax.set_xlabel("Q − QF (Macro-F1, pp)")
    ax.set_title(TASK_LABELS[task], loc="left")
    clean_axis(ax, grid=False)
    ax.grid(True, axis="x", color=PALETTE["grid"], linewidth=0.55)
    panel_label(ax, chr(ord("a") + panel_index))
fig.suptitle("L'addestramento del circuito produce benefici limitati e dipendenti dal regime",
             fontsize=14, fontweight="semibold", y=1.02)
fig.text(0.5, -0.015, "Viola: Q migliore · ambra: circuito congelato QF migliore · media e IC95% bootstrap",
         ha="center", fontsize=8.8, color="#4B5563")
fig.tight_layout()
save_figure(fig, "05_q_minus_qf")


# Figura 6 — Sensibilità a 8−4 qubit

Una heatmap divergente sostituisce le linee a zig-zag. Il colore e il numero annotato
esprimono la stessa quantità, quindi la figura resta leggibile anche stampata.


In [ ]:
qubit_summary = summary(
    qubit_delta, ["task", "subset_size", "model"], "difference_pp"
)
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.4))
models = ("B2", "Q", "QF")
all_values = qubit_summary["estimate"].abs()
color_limit = max(0.5, float(np.nanquantile(all_values, 0.95)))
for panel_index, task in enumerate(("binary", "three_class")):
    panel = qubit_summary.loc[qubit_summary["task"].eq(task)]
    matrix = panel.pivot(index="model", columns="subset_size", values="estimate")
    matrix = matrix.reindex(index=models, columns=REGIMES)
    sns.heatmap(matrix, ax=axes[panel_index], cmap="RdBu_r", center=0,
                vmin=-color_limit, vmax=color_limit, annot=True, fmt="+.2f",
                linewidths=1.0, linecolor="white",
                cbar=panel_index == 1,
                cbar_kws={"label": "8 − 4 qubit (pp)", "shrink": 0.82})
    axes[panel_index].set_title(TASK_LABELS[task], loc="left")
    axes[panel_index].set_xlabel("Immagini di training")
    axes[panel_index].set_ylabel("Modello" if panel_index == 0 else "")
    axes[panel_index].set_xticklabels([REGIME_LABELS[r] for r in REGIMES], rotation=0)
    axes[panel_index].set_yticklabels(models, rotation=0)
    panel_label(axes[panel_index], chr(ord("a") + panel_index))
fig.suptitle("Aumentare da 4 a 8 qubit non determina un miglioramento sistematico",
             fontsize=14, fontweight="semibold", y=1.04)
fig.text(0.5, -0.03, "Valori positivi favoriscono 8 qubit; valori negativi favoriscono 4 qubit",
         ha="center", fontsize=8.8, color="#4B5563")
fig.tight_layout()
save_figure(fig, "06_qubit_sensitivity")


# Figura 7 — Frontiera prestazioni–costo

Sono mostrati soltanto i punti aggregati per regime, collegati nell'ordine
100 → 300 → 900 → Full. Le etichette identificano esclusivamente gli endpoint Full.


In [ ]:
# Ricostruisce i tempi assoluti e li unisce alla Macro-F1 dello stesso seed.
training_costs = costs.loc[costs["cost_metric"].eq("training_seconds")].copy()
cost_parts = []
for model in ("Q", "QF"):
    part = training_costs.loc[training_costs["numerator_model"].eq(model)].copy()
    part["training_seconds"] = pd.to_numeric(part[model], errors="coerce")
    cost_parts.append(part[key + ["training_seconds"]].assign(model=model))
b2_cost = training_costs[key + ["B2"]].drop_duplicates(key).rename(
    columns={"B2": "training_seconds"}
)
cost_parts.append(b2_cost.assign(model="B2"))
absolute_cost = pd.concat(cost_parts, ignore_index=True)
pareto = performance.merge(
    absolute_cost, on=key + ["model"], how="inner", validate="one_to_one"
)
pareto_summary = pareto.groupby(
    ["task", "subset_size", "n_qubits", "model"], as_index=False
).agg(f1_macro=("f1_macro", "mean"),
      training_seconds=("training_seconds", "median"),
      n=("repeat_seed", "nunique"))

try:
    from adjustText import adjust_text
except ImportError:
    def adjust_text(texts, ax=None, **_):
        # Fallback deterministico per ambienti senza adjustText: separa verticalmente le etichette.
        if not texts:
            return texts
        limits = ax.get_ylim()
        minimum_gap = 0.028 * (limits[1] - limits[0])
        ordered = sorted(texts, key=lambda item: item.get_position()[1])
        previous = -np.inf
        for item in ordered:
            x, y = item.get_position()
            y = max(y, previous + minimum_gap)
            item.set_position((x, y))
            previous = y
        return texts

fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.2))
line_styles = {4: "--", 8: "-"}
markers = {4: "o", 8: "D"}
for panel_index, task in enumerate(("binary", "three_class")):
    ax = axes[panel_index]
    panel = pareto_summary.loc[pareto_summary["task"].eq(task)].copy()
    texts = []
    for model in ("B2", "Q", "QF"):
        for qubits in (4, 8):
            rows = panel.loc[panel["model"].eq(model) & panel["n_qubits"].eq(qubits)].copy()
            rows["position"] = rows["subset_size"].map({r: i for i, r in enumerate(REGIMES)})
            rows = rows.sort_values("position")
            ax.plot(rows["training_seconds"], rows["f1_macro"],
                    color=PALETTE[model], linestyle=line_styles[qubits], linewidth=1.7,
                    marker=markers[qubits], markersize=6.3,
                    markeredgecolor="white", markeredgewidth=0.8, alpha=0.92)
            endpoint = rows.loc[rows["subset_size"].eq("full")].iloc[0]
            texts.append(ax.text(endpoint["training_seconds"], endpoint["f1_macro"],
                                 f" {model}-{qubits}", fontsize=7.8,
                                 color=PALETTE[model], va="center"))
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
    ax.set_xlabel("Tempo totale di training (s, scala log)")
    ax.set_ylabel("Macro-F1")
    ax.set_title(TASK_LABELS[task], loc="left")
    clean_axis(ax)
    panel_label(ax, chr(ord("a") + panel_index))
    ax.text(0.03, 0.97, "Direzione desiderabile ↖", transform=ax.transAxes,
            ha="left", va="top", fontsize=8.3, color="#374151")
    adjust_text(texts, ax=ax, only_move={"text": "y"},
                arrowprops={"arrowstyle": "-", "color": "#9CA3AF", "lw": 0.6})

legend = []
for model in ("B2", "Q", "QF"):
    legend.append(Line2D([0], [0], color=PALETTE[model], linewidth=2,
                         label=MODEL_STYLES[model]["label"]))
legend.extend([
    Line2D([0], [0], color="#4B5563", linestyle="--", marker="o", label="4 qubit"),
    Line2D([0], [0], color="#4B5563", linestyle="-", marker="D", label="8 qubit"),
])
fig.legend(handles=legend, loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=5)
fig.suptitle("B2 mantiene la frontiera più favorevole tra prestazioni e costo",
             fontsize=14, fontweight="semibold", y=1.10)
fig.text(0.5, -0.015,
         "Punti: Macro-F1 media e tempo mediano · traiettoria: 100 → 300 → 900 → Full · etichette: endpoint Full",
         ha="center", fontsize=8.8, color="#4B5563")
fig.tight_layout()
save_figure(fig, "07_pareto_performance_cost")


# Esportazione e controllo finale


In [ ]:
captions = {
    "Figura 1": "Macro-F1 di B2, Q e QF a 8 qubit nei quattro regimi. I punti sono le medie sui dieci seed e le bande gli IC95% bootstrap.",
    "Figura 2": "Differenza appaiata Q−B2. La linea tratteggiata indica parità; gli asterischi identificano i confronti con p corretto secondo Holm inferiore a 0,05.",
    "Figura 3": "Differenza-in-differenze [(Q−B2) tre classi − (Q−B2) binario]. Valori negativi indicano che Q perde terreno nel task più complesso.",
    "Figura 4": "F1 della classe merger per B2, Q e QF, separata per configurazioni a 4 e 8 qubit. Le bande rappresentano gli IC95% bootstrap.",
    "Figura 5": "Differenza Q−QF: valori positivi indicano un beneficio dell'addestramento dei parametri quantistici rispetto al circuito congelato.",
    "Figura 6": "Differenza appaiata 8−4 qubit in Macro-F1. La heatmap divergente separa task, modello e regime senza sovrapporre traiettorie.",
    "Figura 7": "Relazione tra tempo totale di training e Macro-F1. Ogni traiettoria collega i regimi 100, 300, 900 e Full; sono etichettati soltanto gli endpoint Full.",
}
captions_path = OUTPUT_DIR / "captions.md"
captions_path.write_text("# Didascalie\n\n" + "\n\n".join(
    f"## {title}\n\n{text}" for title, text in captions.items()
), encoding="utf-8")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_notebook": "04_colab_results_analysis.ipynb",
    "input_directory": str(TABLE_DIR),
    "input_tables": list(REQUIRED_TABLES),
    "figures": EXPORTED_FIGURES,
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__,
                 "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
                 "seaborn": sns.__version__},
    "style": {"palette": PALETTE, "dpi": EXPORT_DPI, "pdf_fonttype": 42},
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

missing_exports = [
    path for item in EXPORTED_FIGURES for path in (item["png"], item["pdf"])
    if not Path(path).is_file() or Path(path).stat().st_size == 0
]
if len(EXPORTED_FIGURES) != 7 or missing_exports:
    raise RuntimeError(
        f"Esportazione incompleta: figure={len(EXPORTED_FIGURES)}, mancanti={missing_exports}"
    )

display(Markdown(
    f"## Restyling completato\n\n- Figure: **{len(EXPORTED_FIGURES)}** coppie PNG/PDF"
    f"\n- Output: `{OUTPUT_DIR}`\n- Input: soli CSV da `{TABLE_DIR}`"
))
